# Supply Chain Optimizer — Agent Layer

Router → SQL Agent (gold Delta tables) or Graph Agent (Neo4j AuraDB)

**Setup:** Store secrets before first run:
```
databricks secrets create-scope supply_chain
databricks secrets put-secret supply_chain anthropic_api_key   --string-value sk-ant-...
databricks secrets put-secret supply_chain neo4j_password       --string-value <password>
```
Or set the widget values below to override.

In [0]:
%pip install anthropic>=0.49.0 neo4j>=5.18.0 --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ── Widget inputs (override secrets if needed) ────────────────────────────────
dbutils.widgets.text("question", "Which suppliers have Critical risk scores?", "Question")
dbutils.widgets.dropdown("route_override", "auto", ["auto", "sql", "graph"], "Force Route")

# ── Credentials ───────────────────────────────────────────────────────────────
import os

def _secret(scope: str, key: str, fallback_env: str = "") -> str:
    try:
        return dbutils.secrets.get(scope=scope, key=key)
    except Exception:
        val = os.getenv(fallback_env, "")
        if not val:
            raise EnvironmentError(
                f"Secret '{scope}/{key}' not found and env var '{fallback_env}' is not set."
            )
        return val

ANTHROPIC_API_KEY = _secret("supply_chain", "anthropic_api_key", "ANTHROPIC_API_KEY")
NEO4J_URI         = os.getenv("NEO4J_URI",      "neo4j+s://26bf512b.databases.neo4j.io")
NEO4J_USERNAME    = os.getenv("NEO4J_USERNAME",  "neo4j")
NEO4J_PASSWORD    = _secret("supply_chain", "neo4j_password", "NEO4J_PASSWORD")
NEO4J_DATABASE    = os.getenv("NEO4J_DATABASE",  "neo4j")

CATALOG        = "supplychain"
SCHEMA         = "supply_chain_medallion"
CLAUDE_MODEL   = "claude-opus-4-6"
MAX_SQL_ROWS   = 500
CACHE_TTL_HOURS = 24

print("Config loaded.")

Config loaded.


In [0]:
# ── SQL helpers (use spark directly — no SDK needed) ──────────────────────────
import json
from typing import Any

from decimal import Decimal

def _safe(val):
    """Convert Spark/Python types that aren't JSON-serializable."""
    if isinstance(val, Decimal):
        return float(val)
    return val

def run_sql(query: str) -> list[dict]:
    """Execute SQL via Spark and return list of dicts."""
    df = spark.sql(query)
    rows = df.limit(MAX_SQL_ROWS).collect()
    cols = df.columns
    return [{k: _safe(v) for k, v in zip(cols, row)} for row in rows]

def ensure_cache_table() -> None:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.answer_cache (
            question_hash  STRING       NOT NULL,
            question_text  STRING,
            query_type     STRING,
            subgraph_type  STRING,
            result_json    STRING,
            computed_at    TIMESTAMP,
            ttl_hours      INT,
            hit_count      BIGINT
        )
        USING DELTA
        CLUSTER BY (question_hash)
        TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
    """)

ensure_cache_table()
print("Cache table ready.")

Cache table ready.


In [0]:
# ── Answer cache ──────────────────────────────────────────────────────────────
import hashlib
from datetime import datetime, timezone

def cache_hash(question: str) -> str:
    return hashlib.sha256(question.strip().lower().encode()).hexdigest()

def cache_get(question: str) -> dict | None:
    h = cache_hash(question)
    rows = run_sql(f"""
        SELECT result_json, computed_at, ttl_hours
        FROM {CATALOG}.{SCHEMA}.answer_cache
        WHERE question_hash = '{h}'
        LIMIT 1
    """)
    if not rows:
        return None
    row = rows[0]
    computed_at = datetime.fromisoformat(str(row["computed_at"]).replace("Z", "+00:00"))
    if computed_at.tzinfo is None:
        computed_at = computed_at.replace(tzinfo=timezone.utc)
    age_hours = (datetime.now(timezone.utc) - computed_at).total_seconds() / 3600
    if age_hours > float(row["ttl_hours"] or 24):
        return None
    # bump hit count
    spark.sql(f"""
        UPDATE {CATALOG}.{SCHEMA}.answer_cache
        SET hit_count = hit_count + 1
        WHERE question_hash = '{h}'
    """)
    try:
        cached = json.loads(row["result_json"])
    except Exception:
        cached = None

    # Validate — delete and recompute if corrupt or failed answer
    answer_text = (cached or {}).get("result", {}).get("answer", "") if cached else ""
    if not cached or "iteration limit" in answer_text.lower():
        h = cache_hash(question)
        spark.sql(f"DELETE FROM {CATALOG}.{SCHEMA}.answer_cache WHERE question_hash = '{h}'")
        return None

    return cached

def cache_put(question: str, result: dict, query_type: str, subgraph_type: str = "") -> None:
    from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, LongType
    from pyspark.sql import Row

    now = datetime.now(timezone.utc).replace(tzinfo=None)  # store as naive UTC timestamp
    schema = StructType([
        StructField("question_hash",  StringType(),    False),
        StructField("question_text",  StringType(),    True),
        StructField("query_type",     StringType(),    True),
        StructField("subgraph_type",  StringType(),    True),
        StructField("result_json",    StringType(),    True),
        StructField("computed_at",    TimestampType(), True),
        StructField("ttl_hours",      IntegerType(),   True),
        StructField("hit_count",      LongType(),      True),
    ])
    row = Row(
        question_hash = cache_hash(question),
        question_text = question,
        query_type    = query_type,
        subgraph_type = subgraph_type or "",
        result_json   = json.dumps(result),
        computed_at   = now,
        ttl_hours     = CACHE_TTL_HOURS,
        hit_count     = 0,
    )
    df = spark.createDataFrame([row], schema=schema)
    df.createOrReplaceTempView("_cache_upsert")
    spark.sql(f"""
        MERGE INTO {CATALOG}.{SCHEMA}.answer_cache AS t
        USING _cache_upsert AS s ON t.question_hash = s.question_hash
        WHEN MATCHED THEN UPDATE SET
            t.result_json = s.result_json,
            t.computed_at = s.computed_at,
            t.ttl_hours   = s.ttl_hours,
            t.hit_count   = 0
        WHEN NOT MATCHED THEN INSERT *
    """)

print("Cache helpers ready.")

Cache helpers ready.


In [0]:
# ── Prompts & tool schemas ────────────────────────────────────────────────────

ROUTER_SYSTEM = """You are a supply chain analytics router.
Call exactly one tool — never answer in plain text.

Use route_to_sql for:
- Risk scores, risk tiers, reliability rankings
- Part availability, stock status, lead times
- Purchase order status, aging, exposure
- Shipment delays, disruption severity
- BOM cost rollup, component counts

Use route_to_graph for:
- "What happens if X fails / is disrupted / goes down"
- "What is at risk if..."
- "Which parts/assemblies depend on supplier X"
- Relationship chains, multi-hop dependencies
- Network centrality, single points of failure
- Ripple-effect or cascading impact analysis
- "Shortest path", "connected to", "linked to"

When in doubt between SQL and graph: if the question involves IMPACT or DEPENDENCY, use graph."""

ROUTER_TOOLS = [
    {
        "name": "route_to_sql",
        "description": "Route to SQL agent for tabular/aggregation questions from gold Delta tables.",
        "input_schema": {
            "type": "object",
            "properties": {
                "question":        {"type": "string"},
                "relevant_tables": {"type": "array", "items": {"type": "string"}},
            },
            "required": ["question", "relevant_tables"],
        },
    },
    {
        "name": "route_to_graph",
        "description": "Route to graph agent for relationship traversal or network analysis.",
        "input_schema": {
            "type": "object",
            "properties": {
                "question":      {"type": "string"},
                "subgraph_type": {
                    "type": "string",
                    "enum": ["supplier_risk", "bom_dependency", "shipment_route", "full_network"],
                },
            },
            "required": ["question", "subgraph_type"],
        },
    },
]

SQL_AGENT_SYSTEM = f"""You are a Databricks SQL analyst for a manufacturing supply chain.
Gold tables in {CATALOG}.{SCHEMA}:
- gold_supplier_risk: supplier_id, name, country, tier, risk_score (0-100), risk_tier (Low/Medium/High/Critical)
- gold_part_availability: part_id, part_name, category, is_critical, facility_id, stock_status
- gold_active_purchase_orders: po_id, supplier_id, part_id, status, aging_days, exposure_score
- gold_shipment_pipeline: shipment_id, carrier, delay_days, disruption_severity, route_key
- gold_bom_explosion: top_parent_part_id, component_part_id, depth (1-2), cumulative_quantity, rolled_up_cost_usd

Always use fully-qualified table names: {CATALOG}.{SCHEMA}.<table>
Return concise, actionable results (max {MAX_SQL_ROWS} rows)."""

SQL_AGENT_TOOLS = [{
    "name": "execute_sql",
    "description": "Execute SQL against Databricks gold tables.",
    "input_schema": {
        "type": "object",
        "properties": {
            "sql":         {"type": "string"},
            "description": {"type": "string"},
        },
        "required": ["sql", "description"],
    },
}]

GRAPH_AGENT_SYSTEM = """You are a Neo4j graph analyst for a supply chain network.

Subgraph types and what they contain — you MUST project the right one before querying:

- supplier_risk   → (:Supplier)-[:SUPPLIES]->(:Part)
                    Use for: supplier impact, which parts a supplier provides, risk by supplier
- bom_dependency  → (:Part)-[:REQUIRES {depth}]->(:Part)
                    Use for: BOM chains, component dependencies, assembly breakdown
- shipment_route  → (:Facility)-[:SHIPS_TO]->(:Facility), (:Shipment)-[:DEPARTS_FROM/ARRIVES_AT]->(:Facility)
                    Use for: route disruptions, carrier analysis, facility connections
- full_network    → all of the above combined

Rules:
- For questions involving BOTH a supplier AND downstream parts/assemblies: project BOTH supplier_risk AND bom_dependency (call project_subgraph twice)
- Always verify nodes exist with a COUNT query before running path queries
- When a relationship type is missing, you likely need to project the subgraph that contains it"""

GRAPH_AGENT_TOOLS = [
    {
        "name": "project_subgraph",
        "description": "Project subgraph from Delta into Neo4j (must run before querying).",
        "input_schema": {
            "type": "object",
            "properties": {
                "subgraph_type": {
                    "type": "string",
                    "enum": ["supplier_risk", "bom_dependency", "shipment_route", "full_network"],
                },
            },
            "required": ["subgraph_type"],
        },
    },
    {
        "name": "run_cypher",
        "description": "Run a Cypher query against Neo4j.",
        "input_schema": {
            "type": "object",
            "properties": {
                "cypher":      {"type": "string"},
                "description": {"type": "string"},
            },
            "required": ["cypher", "description"],
        },
    },
]

print("Prompts ready.")

Prompts ready.


In [ ]:
# ── Neo4j connector ───────────────────────────────────────────────────────────
from neo4j import GraphDatabase

class Neo4jConnector:
    BATCH_SIZE = 500

    def __init__(self):
        self._driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
        self._projected: set[str] = set()
        self._ensure_constraints()

    def close(self):
        self._driver.close()

    def query(self, cypher: str, params: dict = None) -> list[dict]:
        from neo4j.graph import Node, Relationship, Path

        def _convert(val):
            if isinstance(val, Node):
                return {"_labels": list(val.labels), **dict(val)}
            elif isinstance(val, Relationship):
                return {"_type": val.type, **dict(val)}
            elif isinstance(val, Path):
                return [_convert(n) for n in val.nodes]
            elif isinstance(val, list):
                return [_convert(v) for v in val]
            return val

        with self._driver.session(database=NEO4J_DATABASE) as s:
            return [{k: _convert(v) for k, v in dict(r).items()} for r in s.run(cypher, params or {})]

    def _write_batch(self, rows: list[dict], cypher: str) -> int:
        if not rows:
            return 0
        total = 0
        with self._driver.session(database=NEO4J_DATABASE) as s:
            for i in range(0, len(rows), self.BATCH_SIZE):
                batch = rows[i : i + self.BATCH_SIZE]
                s.run(cypher, {"rows": batch})
                total += len(batch)
        return total

    def _ensure_constraints(self):
        for cypher in [
            "CREATE CONSTRAINT supplier_id IF NOT EXISTS FOR (s:Supplier) REQUIRE s.id IS UNIQUE",
            "CREATE CONSTRAINT part_id     IF NOT EXISTS FOR (p:Part)     REQUIRE p.id IS UNIQUE",
            "CREATE CONSTRAINT facility_id IF NOT EXISTS FOR (f:Facility) REQUIRE f.id IS UNIQUE",
            "CREATE CONSTRAINT shipment_id IF NOT EXISTS FOR (sh:Shipment) REQUIRE sh.id IS UNIQUE",
        ]:
            try:
                self.query(cypher)
            except Exception:
                pass

    _EXISTENCE_CHECKS = {
        "supplier_risk":  "MATCH ()-[:SUPPLIES]->() RETURN count(*) AS n",
        "bom_dependency": "MATCH ()-[:REQUIRES]->() RETURN count(*) AS n",
        "shipment_route": "MATCH ()-[:SHIPS_TO]->()  RETURN count(*) AS n",
    }

    def _already_projected(self, subgraph_type: str) -> bool:
        if subgraph_type == "full_network":
            return all(self._already_projected(t) for t in ["supplier_risk", "bom_dependency", "shipment_route"])
        cypher = self._EXISTENCE_CHECKS.get(subgraph_type)
        if not cypher:
            return False
        rows = self.query(cypher)
        return bool(rows and rows[0].get("n", 0) > 0)

    def project_subgraph(self, subgraph_type: str) -> int:
        if subgraph_type in self._projected or self._already_projected(subgraph_type):
            self._projected.add(subgraph_type)
            print(f"  [Neo4j] '{subgraph_type}' already projected — skipping")
            return 0
        projectors = {
            "supplier_risk":   self._project_supplier_risk,
            "bom_dependency":  self._project_bom_dependency,
            "shipment_route":  self._project_shipment_route,
            "full_network":    self._project_full_network,
        }
        if subgraph_type not in projectors:
            raise ValueError(f"Unknown subgraph_type: {subgraph_type}")
        count = projectors[subgraph_type]()
        self._projected.add(subgraph_type)
        print(f"  [Neo4j] Projected '{subgraph_type}': {count} nodes/rels")
        return count

    def _project_supplier_risk(self) -> int:
        total = 0
        rows = run_sql(f"SELECT DISTINCT supplier_id, supplier_name AS name, country, tier, reliability_score, risk_score, risk_tier FROM {CATALOG}.{SCHEMA}.gold_supplier_risk LIMIT 500")
        total += self._write_batch(rows, """
            UNWIND $rows AS r
            MERGE (s:Supplier {id: r.supplier_id})
            SET s.name=r.name, s.country=r.country, s.tier=r.tier,
                s.reliability_score=toFloat(r.reliability_score),
                s.risk_score=toFloat(r.risk_score), s.risk_tier=r.risk_tier
        """)
        rows = run_sql(f"SELECT DISTINCT part_id, part_name AS name, category, is_critical FROM {CATALOG}.{SCHEMA}.gold_part_availability LIMIT 500")
        total += self._write_batch(rows, """
            UNWIND $rows AS r
            MERGE (p:Part {id: r.part_id})
            SET p.name=r.name, p.category=r.category, p.is_critical=r.is_critical
        """)
        rows = run_sql(f"""
            SELECT supplier_id, part_id, COUNT(*) AS po_count, AVG(age_days) AS avg_delay_days
            FROM {CATALOG}.{SCHEMA}.gold_active_purchase_orders
            GROUP BY supplier_id, part_id LIMIT 2000
        """)
        total += self._write_batch(rows, """
            UNWIND $rows AS r
            MATCH (s:Supplier {id: r.supplier_id})
            MATCH (p:Part {id: r.part_id})
            MERGE (s)-[rel:SUPPLIES]->(p)
            SET rel.po_count=toInteger(r.po_count), rel.avg_delay_days=toFloat(r.avg_delay_days)
        """)
        return total

    def _project_bom_dependency(self) -> int:
        total = 0
        rows = run_sql(f"""
            SELECT top_parent_part_id AS part_id, top_parent_name AS name, top_parent_category AS category FROM {CATALOG}.{SCHEMA}.gold_bom_explosion
            UNION SELECT component_part_id, component_name, component_category FROM {CATALOG}.{SCHEMA}.gold_bom_explosion
            LIMIT 1000
        """)
        total += self._write_batch(rows, """
            UNWIND $rows AS r MERGE (p:Part {id: r.part_id}) SET p.name=r.name, p.category=r.category
        """)
        rows = run_sql(f"SELECT top_parent_part_id AS parent_id, component_part_id AS child_id, cumulative_quantity, depth FROM {CATALOG}.{SCHEMA}.gold_bom_explosion LIMIT 5000")
        total += self._write_batch(rows, """
            UNWIND $rows AS r
            MATCH (parent:Part {id: r.parent_id}) MATCH (child:Part {id: r.child_id})
            MERGE (parent)-[rel:REQUIRES {depth: toInteger(r.depth)}]->(child)
            SET rel.cumulative_quantity=toFloat(r.cumulative_quantity)
        """)
        return total

    def _project_shipment_route(self) -> int:
        total = 0
        rows = run_sql(f"""
            SELECT origin_facility_id AS facility_id FROM {CATALOG}.{SCHEMA}.gold_shipment_pipeline
            UNION SELECT destination_facility_id FROM {CATALOG}.{SCHEMA}.gold_shipment_pipeline LIMIT 200
        """)
        total += self._write_batch(rows, "UNWIND $rows AS r MERGE (f:Facility {id: r.facility_id})")
        rows = run_sql(f"SELECT shipment_id, carrier, status, delay_days, disruption_severity, origin_facility_id, destination_facility_id, route_key FROM {CATALOG}.{SCHEMA}.gold_shipment_pipeline LIMIT 2000")
        total += self._write_batch(rows, """
            UNWIND $rows AS r
            MERGE (shp:Shipment {id: r.shipment_id})
            SET shp.carrier=r.carrier, shp.status=r.status,
                shp.delay_days=toInteger(r.delay_days), shp.disruption_severity=r.disruption_severity
            WITH shp, r
            MATCH (o:Facility {id: r.origin_facility_id})
            MATCH (d:Facility {id: r.destination_facility_id})
            MERGE (shp)-[:DEPARTS_FROM]->(o) MERGE (shp)-[:ARRIVES_AT]->(d)
            MERGE (o)-[rt:SHIPS_TO {carrier: r.carrier}]->(d) SET rt.route_key=r.route_key
        """)
        return total

    def _project_full_network(self) -> int:
        return self._project_supplier_risk() + self._project_bom_dependency() + self._project_shipment_route()

neo4j = Neo4jConnector()
print("Neo4j connector ready.")

In [ ]:
# ── SQL Agent ─────────────────────────────────────────────────────────────────
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def _extract_text(content: list) -> str:
    return "\n".join(b.text for b in content if hasattr(b, "type") and b.type == "text").strip()

def sql_agent_answer(question: str, relevant_tables: list[str]) -> dict:
    messages = [{"role": "user", "content": question}]
    executed_queries = []

    for _ in range(6):
        response = client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=4096,
            thinking={"type": "adaptive"},
            system=SQL_AGENT_SYSTEM,
            tools=SQL_AGENT_TOOLS,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            return {"answer": _extract_text(response.content), "queries": executed_queries, "relevant_tables": relevant_tables}

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue
                sql  = block.input.get("sql", "")
                desc = block.input.get("description", "")
                print(f"  [SQL] {desc}\n{sql}")
                try:
                    rows = run_sql(sql)
                    executed_queries.append({"sql": sql, "description": desc, "rows": rows})
                    tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(rows[:MAX_SQL_ROWS])})
                except Exception as e:
                    tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": f"Error: {e}", "is_error": True})
            messages.append({"role": "user", "content": tool_results})

    return {"answer": "Could not complete within iteration limit.", "queries": executed_queries, "relevant_tables": relevant_tables}

In [ ]:
# ── Graph Agent ───────────────────────────────────────────────────────────────

def graph_agent_answer(question: str, subgraph_type: str) -> dict:
    _required = {
        "supplier_risk":  ["supplier_risk"],
        "bom_dependency": ["bom_dependency"],
        "shipment_route": ["shipment_route"],
        "full_network":   ["supplier_risk", "bom_dependency", "shipment_route"],
    }
    for sg in _required.get(subgraph_type, [subgraph_type]):
        neo4j.project_subgraph(sg)

    messages = [{"role": "user", "content": question}]
    cypher_queries = []
    nodes_projected = None

    for _ in range(8):
        response = client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=4096,
            thinking={"type": "adaptive"},
            system=GRAPH_AGENT_SYSTEM,
            tools=GRAPH_AGENT_TOOLS,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            return {"answer": _extract_text(response.content), "cypher_queries": cypher_queries, "subgraph_type": subgraph_type, "nodes_projected": nodes_projected}

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue
                if block.name == "project_subgraph":
                    sg = block.input.get("subgraph_type", subgraph_type)
                    try:
                        count = neo4j.project_subgraph(sg)
                        nodes_projected = count
                        result_str = json.dumps({"status": "projected", "subgraph_type": sg, "nodes_created": count})
                    except Exception as e:
                        result_str = json.dumps({"error": str(e)})
                    tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result_str})

                elif block.name == "run_cypher":
                    cypher = block.input.get("cypher", "")
                    desc   = block.input.get("description", "")
                    print(f"  [Cypher] {desc}\n{cypher}")
                    try:
                        rows = neo4j.query(cypher)
                        cypher_queries.append({"cypher": cypher, "description": desc, "rows": rows})
                        tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(rows[:50])})
                    except Exception as e:
                        tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": f"Error: {e}", "is_error": True})
            messages.append({"role": "user", "content": tool_results})

    return {"answer": "Could not complete within iteration limit.", "cypher_queries": cypher_queries, "subgraph_type": subgraph_type, "nodes_projected": nodes_projected}

In [0]:
# ── Router ────────────────────────────────────────────────────────────────────

def route_and_answer(question: str, route_override: str = "auto") -> dict:
    # Cache check
    cached = cache_get(question)
    if cached:
        print("  [Cache HIT]")
        return {**cached, "from_cache": True}

    # Classify
    if route_override in ("sql", "graph"):
        route = route_override
        tool_input = {
            "question": question,
            "relevant_tables": [] if route_override == "sql" else None,
            "subgraph_type": "full_network" if route_override == "graph" else None,
        }
        print(f"  [Route FORCED → {route.upper()}]")
    else:
        response = client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=512,
            system=ROUTER_SYSTEM,
            tools=ROUTER_TOOLS,
            tool_choice={"type": "any"},
            messages=[{"role": "user", "content": question}],
        )
        tool_block = next((b for b in response.content if b.type == "tool_use"), None)
        if tool_block is None:
            route, tool_input = "sql", {"question": question, "relevant_tables": []}
        else:
            route = "sql" if tool_block.name == "route_to_sql" else "graph"
            tool_input = tool_block.input
        print(f"  [Route → {route.upper()}]")

    # Delegate
    if route == "sql":
        result = sql_agent_answer(question, tool_input.get("relevant_tables", []))
    else:
        result = graph_agent_answer(question, tool_input.get("subgraph_type", "full_network"))

    payload = {"route": route, "result": result, "from_cache": False}
    # Only cache successful answers
    answer_text = result.get("answer", "")
    if answer_text and "iteration limit" not in answer_text.lower():
        cache_put(question, payload, route, tool_input.get("subgraph_type", ""))
    return payload

print("Router ready.")

Router ready.


In [0]:
# ── Ask a question ────────────────────────────────────────────────────────────
question       = dbutils.widgets.get("question")
route_override = dbutils.widgets.get("route_override")

print(f"\nQuestion: {question}\n")
output = route_and_answer(question, route_override)

route      = output.get("route", "?")
from_cache = output.get("from_cache", False)
inner      = output.get("result", {})

print(f"\n{'='*60}")
print(f"  Route: {route.upper()}{'  [CACHED]' if from_cache else ''}")
print(f"{'='*60}")
print(inner.get("answer", ""))

queries = inner.get("queries") or inner.get("cypher_queries") or []
if queries:
    print(f"\n  — {len(queries)} quer{'y' if len(queries)==1 else 'ies'} executed")


Question: Show me the most depended-upon components in the BOM network

  [Route → GRAPH]
  [Neo4j] 'bom_dependency' already projected — skipping
  [Neo4j] 'bom_dependency' already projected — skipping
  [Cypher] Find the most depended-upon components in the BOM network by counting how many distinct parts require each component.

  Route: GRAPH
Here are the **most depended-upon components** in the BOM network — these are the parts that the greatest number of other parts/assemblies require:

| Rank | Component ID | Component Name | # of Dependents |
|------|-------------|----------------|-----------------|
| 1 | PRT-00388 | Mounting Bracket #389 | 3 |
| 2 | PRT-00156 | Titanium Rod #157 | 3 |
| 3 | PRT-00196 | Hydraulic Actuator #197 | 3 |
| 4 | PRT-00222 | Carbon Fiber Roll #223 | 3 |
| 5 | PRT-00130 | Rubber Compound #131 | 3 |
| 6 | PRT-00117 | Magnesium Sheet #118 | 3 |
| 7 | PRT-00235 | Cooling Module #236 | 3 |
| 8 | PRT-00209 | Aluminum Ingot #210 | 3 |
| 9 | PRT-00248 | Power S

## Example questions to try

**SQL (risk & operations)**
- `Which suppliers have Critical risk scores?`
- `Show me all delayed purchase orders over 30 days old`
- `What are the top 10 most expensive BOM assemblies?`
- `Which shipments have High or Critical disruption severity?`
- `What is the stock status for critical parts?`

**Graph (relationships & impact)**
- `What parts are at risk if our highest-risk supplier fails?`
- `Which critical parts have only a single supplier?`
- `What assemblies would be affected if a Tier-1 supplier from China is disrupted?`
- `Show me the most depended-upon components in the BOM network`
- `Which carrier routes have the most disrupted shipments?`